# Sewage on the Dart: How Bad Is It Really, and How Does It Compare?

*BEE2041 Empirical Project*

---
> *here: explain the general logic i.e. paddle the dart>want to know whether the water is full of sewage>if so, how bad is the problem>relative to other rivers and other water companies across the country for a better understanding*

---

## Section 0 — Setup

The cell below imports all libraries used in this notebook and sets some plot styles and output formatting.

In [6]:
import os # for interacting with the computer's file system e.g. to create folders or file paths for the correct OS (i.e. fixes / vs \ between mac/linux and windows)
import time # allows us to use things like time.sleep() later
import warnings # allows us to do things like suppressing warning messages (done later in this code block)
import zipfile # allows us to read and extract ZIP  files
import io # for handling data streams (used later to read ZIP contents in memory)

import numpy as np # for numerical operations
import pandas as pd # necessary for data manipulation - dataframes used later are from this

import requests #  lets Python make HTTP requests (allows us to download data and query APIs later)

import matplotlib.pyplot as plt # library for python plotting
import matplotlib.ticker as mticker # allows for control of axis formatting
import seaborn as sns # a better python plotting library atop matplot (better looking plots ect.)
import folium # library for creating interacive maps (to use later to map sewage overflow locations)

from IPython.display import display # allows outputs (specifically dataframes) to be nicely formatted in outputs

warnings.filterwarnings('ignore') # suppresses some warning messages for minor problems (avoids messy outputs from unnecessary error messages)

sns.set_theme(style='whitegrid', font_scale=1.1) # sets global seaborn plot styling 
COLOUR_DART      = '#1f6aa5'   # blue — used for Dart-specific highlights
COLOUR_SWW       = '#e07b39'   # orange — used for other SWW rivers
COLOUR_NATIONAL  = '#6c757d'   # grey — national/other companies

print('Setup complete.')

Setup complete.


---

## Section 1 — Fetch EDM Storm Overflow Data

### What is the EDM dataset?
> *[explain here what the EDM dataset is - the environment agency requires water companies to fit discharge monitors on storm overflows.]*
> reference the dataset somewhere properly to avoid copyright ect.

> here: explain what the cell does - requests the excel files of this publicly published data for each year>saves them to a raw data file locally>ensures files aren't re-downloaded when running the code again.
> here: explain the logic of this method - rather than manually downloading the files>retreive them programmatically (even if more faffy at the time)>avoids manual steps>ensures better reproducibility."
> here: explain 2020 being (at least temporarily) dropped - 2020 has separate per-company Excel files rather than combined file>inconsistent with other years>overcomplicates what would be required>dropped for ease (is oldest data anyway)> allows for pipeline to be clearer and built quicker>come back to add 2020 after the fact (if time allows)

In [7]:
# creates dictionary mapping year to the URL where the zip folder holds that year's data
EDM_URLS = {
    2021: 'https://environment.data.gov.uk/api/file/download?fileDataSetId=c55e170e-3c75-49a5-8026-a961ff94c8e0&fileName=EDM_2021_Storm_Overflow_Annual_Return.zip',
    2022: 'https://environment.data.gov.uk/api/file/download?fileDataSetId=c55e170e-3c75-49a5-8026-a961ff94c8e0&fileName=EDM_2022_Storm_Overflow_Annual_Return.zip',
    2023: 'https://environment.data.gov.uk/api/file/download?fileDataSetId=c55e170e-3c75-49a5-8026-a961ff94c8e0&fileName=EDM_2023_Storm_Overflow_Annual_Return.zip',
    2024: 'https://environment.data.gov.uk/api/file/download?fileDataSetId=c55e170e-3c75-49a5-8026-a961ff94c8e0&fileName=EDM_2024_Storm_Overflow_Annual_Return.zip',}

RAW_DIR = os.path.join('data', 'raw') # builds the path (with correct slash directions) to data/raw/ where extracted Excel files will be saved.
os.makedirs(RAW_DIR, exist_ok=True) # creates the directory/folder (if it doesnt already exist)

def download_and_extract_edm(year, url, raw_dir): # defines a function which takes 3 inputs (year, url, and raw directory)
    
    # defines clean output filenames for the two Excel files we extract from each ZIP.
    # edm_main_YYYY.xlsx = all companies combined (used for SWW and Dart analysis)
    # edm_summary_YYYY.xlsx = national company-level totals (used for national comparison)
    main_filename    = f'edm_main_{year}.xlsx'
    summary_filename = f'edm_summary_{year}.xlsx'
    # sets the file paths for each of the files extracted from each zip
    main_filepath    = os.path.join(raw_dir, main_filename)
    summary_filepath = os.path.join(raw_dir, summary_filename)

    # skips the download if both files already exist locally - then lets the user know this has happened and the locations of each file
    if os.path.exists(main_filepath) and os.path.exists(summary_filepath):
        print(f'  {year}: already exists, skipping download.')
        return main_filepath, summary_filepath

    print(f'  {year}: downloading...', flush=True) # flush=True forces python to print the text immediately, so that whats happening in real time can be seen
    response = requests.get(url, timeout=60) # sends an HTTP get request to the URL, saving the response to the response object, allows a 1 minute wait for a response
    response.raise_for_status() # stops and gives a clear error if the server returned a failure code e.g. a 404 error


    zip_filepath = os.path.join(raw_dir, f'edm_{year}_temp.zip') # creates filepath for the temporary zip file the same as we've done before
    # saves the zip file to disk (temporarily) so zipfile can open it
    with open(zip_filepath, 'wb') as f:
        f.write(response.content) # writes the downloaded bytes to the disk (in binary mode)


    with zipfile.ZipFile(zip_filepath) as zf:
        all_files = zf.namelist() # returns all file paths inside the zip

        for file_path in all_files: # loops through each of the file paths within the zip
            # split('/') will split the string at every / (i.e. between levels within the filepath), meaning the final ([-1]) string after this spit is simply the filename
            # .lower() makes the matching case-insensitive.
            filename_only = file_path.split('/')[-1].lower()

            # identifies the main all-companies excel file and saves it.
            if 'all water and sewerage' in filename_only and filename_only.endswith('.xlsx'):
                with zf.open(file_path) as source, open(main_filepath, 'wb') as target:
                    target.write(source.read()) # copy contents from zip to disk (in binary mode)

            # identifies the summary excel file and saves it.
            elif 'summary data' in filename_only and filename_only.endswith('.xlsx'):
                with zf.open(file_path) as source, open(summary_filepath, 'wb') as target:
                    target.write(source.read()) # copy contents from zip to disk (in binary mode)
                    
    # delete the temporary zip file now that we have extracted everything we want - keeping our folder clean
    os.remove(zip_filepath) # delete the temporary zip file now that we have extracted everything we want - keeping our folder clean
    
    # verifies both files were successfully extracted - raises errors if they weren't
    if not os.path.exists(main_filepath):
        raise FileNotFoundError(f'Could not find main all-companies Excel file in ZIP for {year}')
    if not os.path.exists(summary_filepath):
        raise FileNotFoundError(f'Could not find summary Excel file in ZIP for {year}')

    print('done.')
    time.sleep(0.5) # brief pause between server requests for politeness
    return main_filepath, summary_filepath


# Loop through all four years, calling the function for each.
# Store results in two dictionaries indexed by year for easy lookup in later cells:
# edm_main_paths[2024]    → 'data/raw/edm_main_2024.xlsx'
# edm_summary_paths[2024] → 'data/raw/edm_summary_2024.xlsx'
print('Downloading and extracting EDM files...')
# creates two empty dictionaries before the loop starts
edm_main_paths    = {}
edm_summary_paths = {}

for year, url in EDM_URLS.items():
    main_path, summary_path  = download_and_extract_edm(year, url, RAW_DIR) # calls the above function - saving the two file paths it returns
    # adds an entry to each dictionary for the current year
    edm_main_paths[year]     = main_path
    edm_summary_paths[year]  = summary_path

# prints a confirmation of what was downloaded - to visually verify everything worked
print('\nAll EDM files ready.')
print('\nMain files:')
for year, path in edm_main_paths.items(): 
    print(f'  {year}: {path}')
print('\nSummary files:')
for year, path in edm_summary_paths.items():
    print(f'  {year}: {path}')

  2021: downloading...
done.
  2022: downloading...
done.
  2023: downloading...
done.
  2024: downloading...
done.

All EDM files ready.

Main files:
  2021: data\raw\edm_main_2021.xlsx
  2022: data\raw\edm_main_2022.xlsx
  2023: data\raw\edm_main_2023.xlsx
  2024: data\raw\edm_main_2024.xlsx

Summary files:
  2021: data\raw\edm_summary_2021.xlsx
  2022: data\raw\edm_summary_2022.xlsx
  2023: data\raw\edm_summary_2023.xlsx
  2024: data\raw\edm_summary_2024.xlsx
